In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import sqlite3
import matplotlib.pyplot as plt


### Million Song Subset Metadata

In [66]:
mss_metadata = pd.read_csv("MillionSongSubsetMetaData.csv")
mss_metadata.head()

,track_id,song_id,artist_id,artist_name,artist_mbid,audio_md5,song_title,year,release,artist_familiarity,...,artist_mbtags,duration,danceability,energy,loudness,tempo,key,time_signature,end_of_fade_in,start_of_fade_out
0,TRARRZU128F4253CA2,SOGSMXL12A81C23D88,AREJXK41187B9A4ACC,Raphaël,c43bb0d6-94d7-410f-80fb-e5a243b18d23,d8bafd4a65d1855aec08991c8b013dc1,Je Sais Que La Terre Est Plate,2008,Je Sais Que La Terre Est Plate (Deluxe),0.557460,...,NaN,148.74077,0.0,0.0,-9.636,124.059,0,4,0.192,141.607
1,TRARRJL128F92DED0E,SOMBCOW12AAF3B229F,AR2XRFQ1187FB417FE,Julie Zenatti,a69cd724-2f57-4ed0-bfed-ba20401eb84c,55f60c97280172e9276723c06e531996,On Efface,2004,Comme Vous,0.626958,...,NaN,252.99546,0.0,0.0,-11.061,80.084,1,4,0.514,241.424
2,TRARRUZ128F9307C57,SOEYIHF12AB017B5F4,ARODOO01187FB44F4A,The Baltimore Consort,60bd8a1c-c093-4849-8f28-08101ca059b1,053fb50807248bef996e6c7a5fe93533,Howells Delight,0,Watkins Ale - Music of the English Renaissance,0.425724,...,NaN,78.02730,0.0,0.0,-24.140,54.874,3,4,0.974,78.027
3,TRARRWA128F42A0195,SODJYEC12A8C13D757,ARJGW911187FB586CA,I Hate Sally,44b5b950-2ae2-403a-8c67-82d8fc72033d,1637df8efe4d89507b6f4ef89a82cacf,Martha Served,2007,Don't Worry Lady,0.611495,...,NaN,163.63057,0.0,0.0,-5.795,77.150,7,3,0.000,158.511
4,TRARRPG12903CD1DE9,SOGSOUE12A58A76443,AR9HQ6Y1187FB3C2CB,Orlando Pops Orchestra,0e6524bd-6641-46a6-bce5-96f06c19aa46,ad4e2031b81e71567fcb7ee46708d255,Zip-A-Dee-Doo-Dah (Song of the South),0,Easy Listening: Cartoon Songs,0.367255,...,NaN,199.99302,0.0,0.0,-16.477,120.382,10,4,0.000,195.808


In [67]:
mss_metadata.isnull().sum()

track_id                 0
song_id                  0
artist_id                0
artist_name              0
artist_mbid            719
audio_md5                0
song_title               1
year                     0
release                  0
artist_familiarity       4
artist_hotttnesss        0
song_hotttnesss       4352
artist_terms             5
artist_mbtags         6290
duration                 0
danceability             0
energy                   0
loudness                 0
tempo                    0
key                      0
time_signature           0
end_of_fade_in           0
start_of_fade_out        0
dtype: int64

In [68]:
# drop columns with too much missing values & irrelevant
mss_metadata.drop(columns=["artist_mbid", "song_hotttnesss", "artist_mbtags"], inplace=True)

In [85]:
tids_unique = mss_metadata['track_id'].unique().tolist()

In [79]:
aid_mb = artist_mbtag_df['artist_id'].unique().tolist()
aid_mss = mss_metadata['artist_id'].unique().tolist()

In [80]:
len(list(set(aid_mb) & set(aid_mss)))

1091

In [93]:
mss_metadata.head()

,track_id,song_id,artist_id,artist_name,audio_md5,song_title,year,release,artist_familiarity,artist_hotttnesss,artist_terms,duration,danceability,energy,loudness,tempo,key,time_signature,end_of_fade_in,start_of_fade_out
0,TRARRZU128F4253CA2,SOGSMXL12A81C23D88,AREJXK41187B9A4ACC,Raphaël,d8bafd4a65d1855aec08991c8b013dc1,Je Sais Que La Terre Est Plate,2008,Je Sais Que La Terre Est Plate (Deluxe),0.557460,0.386152,"chanson, visual kei, hip hop, pop rock, britis...",148.74077,0.0,0.0,-9.636,124.059,0,4,0.192,141.607
1,TRARRJL128F92DED0E,SOMBCOW12AAF3B229F,AR2XRFQ1187FB417FE,Julie Zenatti,55f60c97280172e9276723c06e531996,On Efface,2004,Comme Vous,0.626958,0.434860,"chanson, dance pop, pop rock, soft rock, femal...",252.99546,0.0,0.0,-11.061,80.084,1,4,0.514,241.424
2,TRARRUZ128F9307C57,SOEYIHF12AB017B5F4,ARODOO01187FB44F4A,The Baltimore Consort,053fb50807248bef996e6c7a5fe93533,Howells Delight,0,Watkins Ale - Music of the English Renaissance,0.425724,0.000000,"early music, celtic, mediaeval, folk, christma...",78.02730,0.0,0.0,-24.140,54.874,3,4,0.974,78.027
3,TRARRWA128F42A0195,SODJYEC12A8C13D757,ARJGW911187FB586CA,I Hate Sally,1637df8efe4d89507b6f4ef89a82cacf,Martha Served,2007,Don't Worry Lady,0.611495,0.334520,"post-hardcore, doomcore, metalcore, screamo, g...",163.63057,0.0,0.0,-5.795,77.150,7,3,0.000,158.511
4,TRARRPG12903CD1DE9,SOGSOUE12A58A76443,AR9HQ6Y1187FB3C2CB,Orlando Pops Orchestra,ad4e2031b81e71567fcb7ee46708d255,Zip-A-Dee-Doo-Dah (Song of the South),0,Easy Listening: Cartoon Songs,0.367255,0.311616,"orchestra, musical theater, british, brazil, o...",199.99302,0.0,0.0,-16.477,120.382,10,4,0.000,195.808


In [94]:
mss_metadata.shape

(10000, 20)

In [92]:
# relevant df to merge
track_tags_df.head()

,track_id,track_tags
0,TRPYHPC128F930F9B0,"instrumental, electronic, calm, prda, contempo..."
1,TRJISWW128F42523F5,"indie pop, indie"
2,TRMEZLT128F425F0DB,"emo, post hardcore, hardcore, adorable, scream..."
3,TRQODPB128F934C699,"favorites, Grind, death metal, grindcore, Brut..."
4,TRBDOPE128F422C1EB,"rock, alternative, britpop, songwriter, dark, ..."


In [95]:
track_tags_df.shape

(636334, 2)

In [100]:
track_tags_df[track_tags_df['track_id'] == 'TRARRZU128F4253CA2']

,track_id,track_tags


In [99]:
mss_metadata_withtracktags = pd.merge(mss_metadata, track_tags_df, on='track_id', how='left')
mss_metadata_withtracktags.head()

,track_id,song_id,artist_id,artist_name,audio_md5,song_title,year,release,artist_familiarity,artist_hotttnesss,...,duration,danceability,energy,loudness,tempo,key,time_signature,end_of_fade_in,start_of_fade_out,track_tags
0,TRARRZU128F4253CA2,SOGSMXL12A81C23D88,AREJXK41187B9A4ACC,Raphaël,d8bafd4a65d1855aec08991c8b013dc1,Je Sais Que La Terre Est Plate,2008,Je Sais Que La Terre Est Plate (Deluxe),0.557460,0.386152,...,148.74077,0.0,0.0,-9.636,124.059,0,4,0.192,141.607,NaN
1,TRARRJL128F92DED0E,SOMBCOW12AAF3B229F,AR2XRFQ1187FB417FE,Julie Zenatti,55f60c97280172e9276723c06e531996,On Efface,2004,Comme Vous,0.626958,0.434860,...,252.99546,0.0,0.0,-11.061,80.084,1,4,0.514,241.424,"french, bright voice, menu du jour"
2,TRARRUZ128F9307C57,SOEYIHF12AB017B5F4,ARODOO01187FB44F4A,The Baltimore Consort,053fb50807248bef996e6c7a5fe93533,Howells Delight,0,Watkins Ale - Music of the English Renaissance,0.425724,0.000000,...,78.02730,0.0,0.0,-24.140,54.874,3,4,0.974,78.027,
3,TRARRWA128F42A0195,SODJYEC12A8C13D757,ARJGW911187FB586CA,I Hate Sally,1637df8efe4d89507b6f4ef89a82cacf,Martha Served,2007,Don't Worry Lady,0.611495,0.334520,...,163.63057,0.0,0.0,-5.795,77.150,7,3,0.000,158.511,"post-hardcore, hardcore-punk"
4,TRARRPG12903CD1DE9,SOGSOUE12A58A76443,AR9HQ6Y1187FB3C2CB,Orlando Pops Orchestra,ad4e2031b81e71567fcb7ee46708d255,Zip-A-Dee-Doo-Dah (Song of the South),0,Easy Listening: Cartoon Songs,0.367255,0.311616,...,199.99302,0.0,0.0,-16.477,120.382,10,4,0.000,195.808,NaN


In [102]:
mss_metadata_filtered = mss_metadata_withtracktags.dropna()

In [106]:
mss_metadata_filtered.head()

,track_id,song_id,artist_id,artist_name,audio_md5,song_title,year,release,artist_familiarity,artist_hotttnesss,...,duration,danceability,energy,loudness,tempo,key,time_signature,end_of_fade_in,start_of_fade_out,track_tags
1,TRARRJL128F92DED0E,SOMBCOW12AAF3B229F,AR2XRFQ1187FB417FE,Julie Zenatti,55f60c97280172e9276723c06e531996,On Efface,2004,Comme Vous,0.626958,0.434860,...,252.99546,0.0,0.0,-11.061,80.084,1,4,0.514,241.424,"french, bright voice, menu du jour"
2,TRARRUZ128F9307C57,SOEYIHF12AB017B5F4,ARODOO01187FB44F4A,The Baltimore Consort,053fb50807248bef996e6c7a5fe93533,Howells Delight,0,Watkins Ale - Music of the English Renaissance,0.425724,0.000000,...,78.02730,0.0,0.0,-24.140,54.874,3,4,0.974,78.027,
3,TRARRWA128F42A0195,SODJYEC12A8C13D757,ARJGW911187FB586CA,I Hate Sally,1637df8efe4d89507b6f4ef89a82cacf,Martha Served,2007,Don't Worry Lady,0.611495,0.334520,...,163.63057,0.0,0.0,-5.795,77.150,7,3,0.000,158.511,"post-hardcore, hardcore-punk"
5,TRARRER128F9328521,SOVVDCO12AB0187AF7,ARDPTGD1187B9AD361,Brand X,b5a179fb28efbc6aabb5fbfbee80b1aa,Liquid Time (composition by John Goodsall),0,X Communication : Trilogy II,0.601306,0.363676,...,279.35302,0.0,0.0,-12.474,99.024,9,4,0.433,261.288,"chill, Fusion"
7,TRARROY128F42281F7,SORWTIF12A6D4FAA41,ARJ5BEW1187FB52361,Inoki,0bb8aae991035572e717ea7d80b25a11,Nuovi Re pt. I I (feat. Tek money - Lady Tambler),0,Nobiltà di strada,0.548022,0.440135,...,259.31710,0.0,0.0,-5.050,87.999,1,4,0.223,249.255,


In [107]:
tids_filtred = mss_metadata_filtered['track_id'].unique().tolist()

### Million Song Subset Audio

In [62]:
mss_audio = pd.read_csv("MillionSongSubsetAudioAnalysis.csv")
mss_audio.head()

,track_id,song_id,artist_id,artist_name,song_title,num_sections,section_density,num_segments,segment_density,entropy,...,attack_time_mean,compression_proxy,beat_interval,beat_stability,beat_bar_ratio,bar_interval,bar_stability,tatum_interval,tatum_stability,tatum_beat_ratio
0,TRARRZU128F4253CA2,SOGSMXL12A81C23D88,AREJXK41187B9A4ACC,Raphaël,Je Sais Que La Terre Est Plate,6,0.040339,562,3.778386,2.207899,...,0.052350,7.012726,0.484934,0.009260,4.027397,1.937494,0.017149,0.242467,0.005258,1.996599
1,TRARRJL128F92DED0E,SOMBCOW12AAF3B229F,AR2XRFQ1187FB417FE,Julie Zenatti,On Efface,9,0.035574,774,3.059343,2.395099,...,0.086557,5.327190,0.746450,0.062613,4.826087,3.481281,3.019133,0.248817,0.022639,2.993994
2,TRARRUZ128F9307C57,SOEYIHF12AB017B5F4,ARODOO01187FB44F4A,The Baltimore Consort,Howells Delight,4,0.051264,205,2.627286,2.433510,...,0.097573,5.987556,1.095980,0.025623,4.214286,4.393255,0.053273,0.274002,0.008181,4.050847
3,TRARRWA128F42A0195,SODJYEC12A8C13D757,ARJGW911187FB586CA,I Hate Sally,Martha Served,6,0.036668,469,2.866213,2.477836,...,0.054662,4.040578,0.781573,0.048328,3.075758,2.392370,0.257940,0.260554,0.017127,3.004926
4,TRARRPG12903CD1DE9,SOGSOUE12A58A76443,AR9HQ6Y1187FB3C2CB,Orlando Pops Orchestra,Zip-A-Dee-Doo-Dah (Song of the South),10,0.050002,524,2.620091,2.412606,...,0.101949,4.644727,0.499178,0.013144,4.030928,1.996074,0.040663,0.249590,0.007134,2.002558


In [65]:
mss_audio.isnull().sum()

track_id                 0
song_id                  0
artist_id                0
artist_name              0
song_title               1
num_sections             0
section_density          0
num_segments             0
segment_density          0
entropy                  0
pitch_concerntration     0
energy                   0
top_note_strength        0
pitch_change_rate        0
timbre_variability       0
timbre_energy            0
timbre_change_rate       0
loudness_mean            0
dynamic_variability      0
dynamic_range            0
attack_strength          0
attack_time_mean         0
compression_proxy        0
beat_interval           25
beat_stability          25
beat_bar_ratio          30
bar_interval            30
bar_stability           30
tatum_interval          15
tatum_stability         15
tatum_beat_ratio        25
dtype: int64

In [110]:
mss_audio_filtered = mss_audio[mss_audio['track_id'].isin(tids_filtred)]
mss_audio_filtered = mss_audio_filtered.drop(columns=['song_id', 'artist_id', 'artist_name', 'song_title'])

In [113]:
mss_audio_filtered = mss_audio_filtered.dropna()

In [114]:
mss_audio_filtered.head()

,track_id,num_sections,section_density,num_segments,segment_density,entropy,pitch_concerntration,energy,top_note_strength,pitch_change_rate,...,attack_time_mean,compression_proxy,beat_interval,beat_stability,beat_bar_ratio,bar_interval,bar_stability,tatum_interval,tatum_stability,tatum_beat_ratio
1,TRARRJL128F92DED0E,9,0.035574,774,3.059343,2.395099,0.406325,0.879287,0.398764,0.823324,...,0.086557,5.327190,0.746450,0.062613,4.826087,3.481281,3.019133,0.248817,0.022639,2.993994
2,TRARRUZ128F9307C57,4,0.051264,205,2.627286,2.433510,0.359156,1.069872,0.473600,1.088328,...,0.097573,5.987556,1.095980,0.025623,4.214286,4.393255,0.053273,0.274002,0.008181,4.050847
3,TRARRWA128F42A0195,6,0.036668,469,2.866213,2.477836,0.286949,1.654929,0.549348,1.006095,...,0.054662,4.040578,0.781573,0.048328,3.075758,2.392370,0.257940,0.260554,0.017127,3.004926
5,TRARRER128F9328521,8,0.028638,986,3.529584,2.462202,0.318254,1.425504,0.568989,1.034330,...,0.037420,7.889651,0.606771,0.009060,4.044643,2.427128,0.030175,0.151693,0.002802,3.997792
7,TRARROY128F42281F7,9,0.034707,1078,4.157073,2.448832,0.354976,1.416393,0.599881,1.341510,...,0.053653,9.928359,0.676080,0.030645,4.085106,2.746872,0.233411,0.338040,0.015875,1.997396


In [115]:
mss_metadata_filtered.head()

,track_id,song_id,artist_id,artist_name,audio_md5,song_title,year,release,artist_familiarity,artist_hotttnesss,...,duration,danceability,energy,loudness,tempo,key,time_signature,end_of_fade_in,start_of_fade_out,track_tags
1,TRARRJL128F92DED0E,SOMBCOW12AAF3B229F,AR2XRFQ1187FB417FE,Julie Zenatti,55f60c97280172e9276723c06e531996,On Efface,2004,Comme Vous,0.626958,0.434860,...,252.99546,0.0,0.0,-11.061,80.084,1,4,0.514,241.424,"french, bright voice, menu du jour"
2,TRARRUZ128F9307C57,SOEYIHF12AB017B5F4,ARODOO01187FB44F4A,The Baltimore Consort,053fb50807248bef996e6c7a5fe93533,Howells Delight,0,Watkins Ale - Music of the English Renaissance,0.425724,0.000000,...,78.02730,0.0,0.0,-24.140,54.874,3,4,0.974,78.027,
3,TRARRWA128F42A0195,SODJYEC12A8C13D757,ARJGW911187FB586CA,I Hate Sally,1637df8efe4d89507b6f4ef89a82cacf,Martha Served,2007,Don't Worry Lady,0.611495,0.334520,...,163.63057,0.0,0.0,-5.795,77.150,7,3,0.000,158.511,"post-hardcore, hardcore-punk"
5,TRARRER128F9328521,SOVVDCO12AB0187AF7,ARDPTGD1187B9AD361,Brand X,b5a179fb28efbc6aabb5fbfbee80b1aa,Liquid Time (composition by John Goodsall),0,X Communication : Trilogy II,0.601306,0.363676,...,279.35302,0.0,0.0,-12.474,99.024,9,4,0.433,261.288,"chill, Fusion"
7,TRARROY128F42281F7,SORWTIF12A6D4FAA41,ARJ5BEW1187FB52361,Inoki,0bb8aae991035572e717ea7d80b25a11,Nuovi Re pt. I I (feat. Tek money - Lady Tambler),0,Nobiltà di strada,0.548022,0.440135,...,259.31710,0.0,0.0,-5.050,87.999,1,4,0.223,249.255,


In [116]:
mss_metadata_audio = pd.merge(mss_metadata_filtered, mss_audio_filtered, on='track_id', how='inner')
mss_metadata_audio.shape

(6426, 47)

In [118]:
mss_metadata_audio.to_csv('mss_metadata_audio.csv')

### Last.fm Dataset - Song Tags & Song Similarity
- the official song tag and song similarity dataset
- The Last.fm dataset consists of two kinds of data at the song level: tags and similar songs. 


#### Track Tags

In [86]:
database = "lastfm_tags.db"

# connect to database
conn = sqlite3.connect(database)
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", conn)
tables

,type,name,tbl_name,rootpage,sql
0,table,tags,tags,2,CREATE TABLE tags (tag TEXT)
1,table,tids,tids,3,CREATE TABLE tids (tid TEXT)
2,table,tid_tag,tid_tag,4,"CREATE TABLE tid_tag (tid INT, tag INT, val FL..."


In [87]:
your_query = """
            SELECT *
            FROM tids;
            """
your_query2 = """
            SELECT *
            FROM tags;
            """
your_query3 = """
            SELECT *
            FROM tid_tag;
            """
tids = pd.read_sql(your_query, conn)
tags = pd.read_sql(your_query2, conn)
tid_tag = pd.read_sql(your_query3, conn)

In [83]:
# Use case 2
tid = 'TRCCOFQ128F4285A9E'
print('We get all tags (with value) for track: %s' % tid)
sql = "SELECT tags.tag FROM tid_tag, tids, tags WHERE tags.ROWID=tid_tag.tag AND tid_tag.tid=tids.ROWID and tids.tid='%s'" % tid
res = conn.execute(sql)
data = res.fetchall()
print(data)

We get all tags (with value) for track: TRCCOFQ128F4285A9E
[('rock',), ('elandal pl-tunteellinen',), ('elandal loved',), ('elandal 5star',), ('yhteiskuntakriittinen',), ('tunteellinen',), ('kicking ass sounds',), ('suomeksi',), ('fiilis',), ('Hassisen kone',), ('poliittinen',), ('Fucking masterpiece',), ('5star',), ('Fantastic Track',), ('feeling',), ('Ismo Alanko',), ('loved',), ('suomirock',), ('melankolia',), ('finnish lyrics',), ('Suomi',), ('emotional',), ('melancholy',), ('Awesome',), ('punk rock',), ('finland',), ('political',), ('finnish',), ('punk',)]


In [88]:
tags_tids = []

for tid in tids_unique:
    sql = "SELECT tags.tag FROM tid_tag, tids, tags WHERE tags.ROWID=tid_tag.tag AND tid_tag.tid=tids.ROWID and tids.tid='%s'" % tid
    res = conn.execute(sql)
    data = res.fetchall()
    tags = []
    for tag in data:
        tags.append(tag[0])
    tags_tids.append(", ".join(tags))

In [90]:
track_tags = {'track_id': tids_unique,
              'track_tags': tags_tids}
track_tags_df = pd.DataFrame(track_tags)
track_tags_df.head()

,track_id,track_tags
0,TRPYHPC128F930F9B0,"instrumental, electronic, calm, prda, contempo..."
1,TRJISWW128F42523F5,"indie pop, indie"
2,TRMEZLT128F425F0DB,"emo, post hardcore, hardcore, adorable, scream..."
3,TRQODPB128F934C699,"favorites, Grind, death metal, grindcore, Brut..."
4,TRBDOPE128F422C1EB,"rock, alternative, britpop, songwriter, dark, ..."


In [142]:
conn.close()

#### Track Similarity

In [2]:
database = "lastfm_similars.db"

# connect to database
conn = sqlite3.connect(database)
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", conn)
tables

,type,name,tbl_name,rootpage,sql
0,table,similars_src,similars_src,2,"CREATE TABLE similars_src (tid TEXT, target TEXT)"
1,table,similars_dest_tmp,similars_dest_tmp,3,"CREATE TABLE similars_dest_tmp (tid1 TEXT, tid..."
2,table,similars_dest,similars_dest,4,"CREATE TABLE similars_dest (tid TEXT, target T..."


In [3]:
your_query = """
            SELECT *
            FROM similars_src;
            """
your_query2 = """
            SELECT *
            FROM similars_dest_tmp;
            """
your_query3 = """
            SELECT *
            FROM similars_dest;
            """
similars_src = pd.read_sql(your_query, conn)
similars_dest_tmp = pd.read_sql(your_query2, conn)
similars_dest = pd.read_sql(your_query3, conn)

In [146]:
similars_src.head()

,tid,target
0,TRCCCYE12903CFF0E9,"TRHZRQH128F92F9AC2,0.498053,TRZQUEN12903CBFFBB..."
1,TRCCCPM12903CBEEE5,"TRNRRXT12903CFAD5A,1,TRWVNWV12903CBEEE7,0.5238..."
2,TRCCCFH12903CEBC70,"TRRGGCN128F92E3579,0.646036,TRTVJGV128F424A147..."
3,TRCCCJT128F429FFF6,"TRZSKOT128F429FFFC,1,TRYVKKD12903CEB9E2,1,TRVC..."
4,TRCCCBJ128F4286E6F,"TRHUKZN128F428B2BD,1,TRMMPCY128F4253A20,0.9906..."


In [154]:
similars_dest.head()

,tid,target
0,TRPYHPC128F930F9B0,"TRHVKYA128F4289436,0.00100462,TRFONPG128F92FC0..."
1,TRJISWW128F42523F5,"TRJFNWV128F4224BE2,0.0120832,TRLCUYW128F932938..."
2,TRMEZLT128F425F0DB,"TRXBRDS12903CB2D9A,0.795339,TRMBEIK128F429B092..."
3,TRQODPB128F934C699,"TRJLOPB128F428CFE4,0.0411795,TRMHIHC12903CDF3D..."
4,TRBDOPE128F422C1EB,"TROUHQG128F425BF2B,0.00230093,TRAXWWD128F930A8..."


In [160]:
# EXAMPLE 1
tid = 'TREDTHC128F92D42F0'
print('We get all similar songs (with value) to %s' % tid)
sql = "SELECT target FROM similars_src WHERE tid='%s'" % tid
res = conn.execute(sql)
data = res.fetchone()[0]

print('(total number of similar tracks: %d)' % (len(data.split(','))/2))


print('We get all songs which consider %s as similar' % tid)
sql2 = "SELECT target FROM similars_dest WHERE tid='%s'" % tid
res2 = conn.execute(sql2)
data2 = res2.fetchone()[0]

print('(total number of track where %s is similar: %d)' % (tid, len(data.split(','))/2))

We get all similar songs (with value) to TREDTHC128F92D42F0
(total number of similar tracks: 133)
We get all songs which consider TREDTHC128F92D42F0 as similar
(total number of track where TREDTHC128F92D42F0 is similar: 133)


In [161]:
 # EXAMPLE 5
tid = 'TREDTHC128F92D42F0'
print('We get similars to %s and order them by similarity value' % tid)

sql = "SELECT target FROM similars_dest WHERE tid='%s'" % tid
res = conn.execute(sql)
data = res.fetchone()[0]

data_unpacked = []
for idx, d in enumerate(data.split(',')):
    if idx % 2 == 0:
        pair = [d]
    else:
        pair.append(float(d))
        data_unpacked.append(pair)
# sort
data_unpacked = sorted(data_unpacked, key=lambda x: x[1], reverse=True)


We get similars to TREDTHC128F92D42F0 and order them by similarity value


In [4]:
# dictionary for similar songs + similarity value for each track id 
similars = []

for tid in tids_unique:
    sql = "SELECT target FROM similars_dest WHERE tid='%s'" % tid
    res = conn.execute(sql)
    data = res.fetchone()[0]

    data_unpacked = []
    for idx, d in enumerate(data.split(',')):
        if idx % 2 == 0:
            pair = [d]
        else:
            pair.append(float(d))
            data_unpacked.append(pair)
    # sort
    data_unpacked = sorted(data_unpacked, key=lambda x: x[1], reverse=True)
    similars.append(data_unpacked)

In [6]:
tracks_similar = {"track_id": tids_unique,
                  "similars": similars}

track_similar_df = pd.DataFrame(tracks_similar)
track_similar_df

,track_id,similars
0,TRPYHPC128F930F9B0,"[[TRKZWHG128F930BF84, 0.0178901], [TRZVOIJ128F..."
1,TRJISWW128F42523F5,"[[TRKKXPZ128F42523F9, 1.0], [TRGSDUJ128F42523F..."
2,TRMEZLT128F425F0DB,"[[TRQUNEC128F425F0E9, 1.0], [TRVNXKM128F425F0D..."
3,TRQODPB128F934C699,"[[TROPSSP128F934C69E, 1.0], [TRNSPRN128F934C69..."
4,TRBDOPE128F422C1EB,"[[TRECQCB128F428DA82, 0.364174], [TRXQKZF128F4..."
...,...,...
636329,TROUXOR128F429114C,"[[TRVOYDT128F428CFCF, 0.159137], [TRCCMEY128EF..."
636330,TRAMXMB128F93525F3,"[[TRMGVSH128F93525F0, 1.0], [TRMXIAN128F930C38..."
636331,TRQQCHF128F9342D51,"[[TRTXOMA128F9342D52, 0.88542], [TRDUYJE128F93..."
636332,TRRVPDQ128F4248B7A,"[[TRVRHDR128F4252C9A, 0.00819855]]"


### Artist Terms
- `artist_term` & `artist_mbtags` dataset 

In [72]:
database = "artist_term.db" 

# connect to database
connection_db = sqlite3.connect(database)
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_db)
tables

,type,name,tbl_name,rootpage,sql
0,table,artists,artists,2,CREATE TABLE artists (artist_id text PRIMARY KEY)
1,table,terms,terms,2459,CREATE TABLE terms (term text PRIMARY KEY)
2,table,artist_term,artist_term,2782,"CREATE TABLE artist_term (artist_id text, term..."
3,table,mbtags,mbtags,2783,CREATE TABLE mbtags (mbtag text PRIMARY KEY)
4,table,artist_mbtag,artist_mbtag,2876,"CREATE TABLE artist_mbtag (artist_id text, mbt..."


In [73]:
# artist table
artist_df = pd.read_sql("""
                          SELECT *
                          FROM artists;
                          """, connection_db)
artist_df.head()

,artist_id
0,AR002UA1187B9A637D
1,AR003FB1187B994355
2,AR006821187FB5192B
3,AR009211187B989185
4,AR009SZ1187B9A73F4


In [74]:
# terms (echonest tags)
terms_df = pd.read_sql("""
                          SELECT *
                          FROM terms;
                          """, connection_db)
terms_df.head()

,term
0,00s
1,00s alternative
2,00s country
3,00s indie
4,00s pop


In [75]:
# artist terms (artist can have many terms)
artist_term_df = pd.read_sql("""
                          SELECT *
                          FROM artist_term;
                          """, connection_db)
artist_term_df.head()

,artist_id,term
0,AR002UA1187B9A637D,garage rock
1,AR002UA1187B9A637D,country rock
2,AR002UA1187B9A637D,free jazz
3,AR002UA1187B9A637D,oi
4,AR002UA1187B9A637D,space rock


In [76]:
# mbtags (echonest tags)
mbtags_df = pd.read_sql("""
                          SELECT *
                          FROM mbtags;
                          """, connection_db)
mbtags_df.head()

,mbtag
0,00s
1,00s 10s
2,1 13 165900 150 7672 22647 34612 48720 59280 7...
3,1 7 186240 183 23558 41608 89158 111733 150833...
4,10s


In [77]:
artist_mbtag_df = pd.read_sql("""
                          SELECT *
                          FROM artist_mbtag;
                          """, connection_db)
artist_mbtag_df.head()

,artist_id,mbtag
0,AR002UA1187B9A637D,uk
1,AR002UA1187B9A637D,rock
2,AR002UA1187B9A637D,garage rock
3,AR006821187FB5192B,bass
4,AR00A6H1187FB5402A,detroit


In [18]:
connection_db.close()

### Artist Similarity
- which artist_id is similar to the specified arist_id

In [19]:
database = "artist_similarity.db" 

# connect to database
connection_db = sqlite3.connect(database)
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_db)
tables

,type,name,tbl_name,rootpage,sql
0,table,artists,artists,2,CREATE TABLE artists (artist_id text PRIMARY KEY)
1,table,similarity,similarity,2459,"CREATE TABLE similarity (target text, similar ..."


In [20]:
artists_df_2 = pd.read_sql("""
                          SELECT *
                          FROM artists;
                          """, connection_db)
artists_df_2.head()

,artist_id
0,AR002UA1187B9A637D
1,AR003FB1187B994355
2,AR006821187FB5192B
3,AR009211187B989185
4,AR009SZ1187B9A73F4


In [21]:
similarity_df = pd.read_sql("""
                          SELECT *
                          FROM similarity;
                          """, connection_db)
similarity_df.head()

,target,similar
0,AR002UA1187B9A637D,ARQDOR81187FB3B06C
1,AR002UA1187B9A637D,AROHMXJ1187B989023
2,AR002UA1187B9A637D,ARAGWVR1187B9B749B
3,AR002UA1187B9A637D,AREQVWS1241B9CC0A4
4,AR002UA1187B9A637D,ARHBE351187FB3B0CD


### Metadata

In [7]:
database = "track_metadata.db"  

In [8]:
# connect to database
connection_db = sqlite3.connect(database)
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_db)

In [9]:
tables

,type,name,tbl_name,rootpage,sql
0,table,songs,songs,2,"CREATE TABLE songs (track_id text PRIMARY KEY,..."


In [10]:
your_query = """
            SELECT *
            FROM songs;
            """

metadata_df = pd.read_sql(your_query, connection_db)
print(metadata_df.shape)
print(metadata_df.dtypes)
metadata_df.head()

(1000000, 14)
track_id               object
title                  object
song_id                object
release                object
artist_id              object
artist_mbid            object
artist_name            object
duration              float64
artist_familiarity    float64
artist_hotttnesss     float64
year                    int64
track_7digitalid        int64
shs_perf                int64
shs_work                int64
dtype: object


,track_id,title,song_id,release,artist_id,artist_mbid,artist_name,duration,artist_familiarity,artist_hotttnesss,year,track_7digitalid,shs_perf,shs_work
0,TRMMMYQ128F932D901,Silent Night,SOQMMHC12AB0180CB8,Monster Ballads X-Mas,ARYZTJS1187B98C555,357ff05d-848a-44cf-b608-cb34b5701ae5,Faster Pussy cat,252.05506,0.649822,0.394032,2003,7032331,-1,0
1,TRMMMKD128F425225D,Tanssi vaan,SOVFVAK12A8C1350D9,Karkuteillä,ARMVN3U1187FB3A1EB,8d7ef530-a6fd-4f8f-b2e2-74aec765e0f9,Karkkiautomaatti,156.55138,0.439604,0.356992,1995,1514808,-1,0
2,TRMMMRX128F93187D9,No One Could Ever,SOGTUKN12AB017F4F1,Butter,ARGEKB01187FB50750,3d403d44-36ce-465c-ad43-ae877e65adc4,Hudson Mohawke,138.97098,0.643681,0.437504,2006,6945353,-1,0
3,TRMMMCH128F425532C,Si Vos Querés,SOBNYVR12A8C13558C,De Culo,ARNWYLR1187B9B2F9C,12be7648-7094-495f-90e6-df4189d68615,Yerba Brava,145.05751,0.448501,0.372349,2003,2168257,-1,0
4,TRMMMWA128F426B589,Tangle Of Aspens,SOHSBXH12A8C13B0DF,Rene Ablaze Presents Winter Sessions,AREQDTE1269FB37231,,Der Mystic,514.29832,0.000000,0.000000,0,2264873,-1,0


In [11]:
connection_db.close()

### Lyrics MxM
http://millionsongdataset.com/musixmatch/

%word1,word2,... - list of top words, in popularity order <br>
TID,MXMID,idx:cnt,idx:cnt,... - track ID from MSD, track ID from musiXmatch, <br>
then word index : word count (word index starts at 1!) <br>

More details on the database:
   - it contains two tables, 'words' and 'lyrics'
   - table 'words' has one column: 'word'. Words are entered according
     to popularity, check their ROWID if you want to check their position.
     ROWID is an implicit column in SQLite, it starts at 1.
   - table 'lyrics' contains 5 columns, see below
       - column 'track_id' -> as usual, track id from the MSD
       - column 'mxm_tid' -> track ID from musiXmatch
       - column 'word' -> a word that is also in the 'words' table
       - column 'count' -> word count for the word
       - column 'is_test' -> 0 if this example is from the train set, 1 if test


In [8]:
# mxm matching msd
rows = [] 

with open("mxm_779k_matches.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        rows.append(line.split("<SEP>"))

mxm_matches_df = pd.DataFrame(rows, columns=[
    "msd_tid",
    "msd_artist",
    "msd_title",
    "mxm_tid",
    "mxm_artist",
    "mxm_title"
])

mxm_matches_df.head()

,msd_tid,msd_artist,msd_title,mxm_tid,mxm_artist,mxm_title
0,TRMMMKD128F425225D,Karkkiautomaatti,Tanssi vaan,4418550,Karkkiautomaatti,Tanssi vaan
1,TRMMMRX128F93187D9,Hudson Mohawke,No One Could Ever,8898149,Hudson Mohawke,No One Could Ever
2,TRMMMCH128F425532C,Yerba Brava,Si Vos Querés,9239868,Yerba Brava,Si vos queres
3,TRMMMXN128F42936A5,David Montgomery,"Symphony No. 1 G minor ""Sinfonie Serieuse""/All...",5346741,Franz Berwald,"Symphony No. 1 in G minor ""Sinfonie Sérieuse"":..."
4,TRMMMBB12903CB7D21,Kris Kross,2 Da Beat Ch'yall,2511405,Kris Kross,2 Da Beat Ch'yall


In [9]:
database = "mxm_dataset.db"  

# connect to database
connection_db = sqlite3.connect(database)
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_db)
tables

,type,name,tbl_name,rootpage,sql
0,table,words,words,2,CREATE TABLE words (word TEXT PRIMARY KEY)
1,table,lyrics,lyrics,4,"CREATE TABLE lyrics (track_id, mxm_tid INT, wo..."


In [10]:
your_query = """
            SELECT *
            FROM words;
            """

words_mxm_df = pd.read_sql(your_query, connection_db)
print(words_mxm_df.shape)
words_mxm_df.head()

(5000, 1)


,word
0,i
1,the
2,you
3,to
4,and


In [11]:
your_query = """
            SELECT *
            FROM lyrics;
            """

lyrics_mxm_df = pd.read_sql(your_query, connection_db)
print(lyrics_mxm_df.shape)
lyrics_mxm_df.head()

(19045332, 5)


,track_id,mxm_tid,word,count,is_test
0,TRAAAAV128F421A322,4623710,i,6,0
1,TRAAAAV128F421A322,4623710,the,4,0
2,TRAAAAV128F421A322,4623710,you,2,0
3,TRAAAAV128F421A322,4623710,to,2,0
4,TRAAAAV128F421A322,4623710,and,5,0


In [12]:
connection_db.close()

In [50]:
trackids = mss_metadata['track_id'].unique().tolist()
unique_words = lyrics_mxm_df['word'].unique().tolist()

In [53]:
trackids = mss_metadata['track_id'].unique().tolist()
track_10K = lyrics_mxm_df[lyrics_mxm_df['track_id'].isin(trackids)]
unique_words = track_10K ['word'].unique().tolist()
len(unique_words)

4851

In [ ]:
songlyrics_bow = {}

for w in unique_words:
    songlyrics_bow[w] = []
    for tid in trackids:
        wordcount = lyrics_mxm_df[(lyrics_mxm_df['track_id'] == tid) & (lyrics_mxm_df['word'] == w)]['count']
        if len(wordcount) != 0:
            songlyrics_bow[w].append(wordcount.iloc[0])
        else:
            songlyrics_bow[w].append(0)


In [ ]:
len(trackids)

In [44]:
tid = trackids[0]
lyrics_mxm_df[(lyrics_mxm_df['track_id'] == tid) & (lyrics_mxm_df['word'] == 'the')]['count'].iloc[0]

np.int64(4)

### Unique Tracks Dataset
- song_id, song_name & artist

In [9]:
# import dataset
unique_track_df = pd.read_csv(
    "unique_tracks.txt",
    sep="<SEP>",
    header=None,
    engine='python',
    names=["track_id", "song_id", "artist_name", "song_title"]
)

unique_track_df.head()

,track_id,song_id,artist_name,song_title
0,TRMMMYQ128F932D901,SOQMMHC12AB0180CB8,Faster Pussy cat,Silent Night
1,TRMMMKD128F425225D,SOVFVAK12A8C1350D9,Karkkiautomaatti,Tanssi vaan
2,TRMMMRX128F93187D9,SOGTUKN12AB017F4F1,Hudson Mohawke,No One Could Ever
3,TRMMMCH128F425532C,SOBNYVR12A8C13558C,Yerba Brava,Si Vos Querés
4,TRMMMWA128F426B589,SOHSBXH12A8C13B0DF,Der Mystic,Tangle Of Aspens


In [10]:
mss_metadata_audio = pd.read_csv('mss_metadata_audio.csv')

In [11]:
unique_tid_subset = mss_metadata_audio['track_id'].unique().tolist()
unique_track_df_subset = unique_track_df[unique_track_df['track_id'].isin(unique_tid_subset)]
unique_track_df_subset.shape

(6426, 4)

In [12]:
unique_track_df_subset.to_csv('unique_tracks_subset.csv', index=False)

In [13]:
unique_track_df_subset[unique_track_df_subset.duplicated(subset=["track_id", "song_id"])]

,track_id,song_id,artist_name,song_title


### User Data

In [14]:
unique_track_df_subset = pd.read_csv('unique_tracks_subset.csv')

In [16]:
songid_unique = unique_track_df_subset['song_id'].unique().tolist()

In [4]:
# import dataset
df = pd.read_csv(
    "train_triplets.txt",
    sep="\t",
    header=None,
    names=["user_id", "song_id", "play_count"]
)
df.head()

,user_id,song_id,play_count
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAKIMP12A8C130995,1
1,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOAPDEY12A81C210A9,1
2,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBBMDR12A8C13253B,2
3,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBFNSP12AF72A0E22,1
4,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOBFOVM12A58A7D494,1


In [17]:
df.shape

(48373586, 3)

In [23]:
user_data = df[df['song_id'].isin(songid_unique)].reset_index(drop=True)

In [24]:
user_data.head()

,user_id,song_id,play_count
0,b80344d063b5ccb3212f76538f3d9e43d87dca9e,SOWEZSI12A81C21CE6,1
1,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SODCXXY12AB0187452,2
2,4bd88bfb25263a75bbdd467e74018f4ae570e5df,SOWPAXV12A67ADA046,18
3,b64cdd1a0bd907e5e00b39e345194768e330d652,SOLXDDC12A6701FBFD,1
4,b64cdd1a0bd907e5e00b39e345194768e330d652,SONJBQX12A6D4F8382,4


In [25]:
user_data.shape

(599645, 3)

In [26]:
user_data.to_csv('train_triplets_subset.csv', index=False)